# Gestión de Modelos Preentrenados

En este notebook aprenderemos a:
1. **Guardar y cargar modelos** localmente (trabajar offline)
2. **Usar GPU** para acelerar inferencia
3. **Cuantizar modelos** para reducir memoria

---
## Instalación

In [ ]:
!pip install transformers accelerate bitsandbytes -q

---
# Parte 1: Guardar y Cargar Modelos Localmente

Por defecto, cada vez que usas `from_pretrained()`, el modelo se descarga de Hugging Face Hub. Esto puede ser lento y requiere internet.

**Solución**: Guardar el modelo localmente y cargarlo desde ahí.

### Ejemplo: Guardar un modelo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

#1. Cargar modelo desde Hugging Face Hub
modelo_nombre = "distilbert-base-uncased-finetuned-sst-2-english"
print(f"Descargando modelo: {modelo_nombre}")

tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
model = AutoModelForSequenceClassification.from_pretrained(modelo_nombre)

print(f"Modelo cargado: {model.config.architectures}")

#2. Guardar localmente
ruta_local = "./modelos/mi_clasificador"
os.makedirs(ruta_local, exist_ok=True)

tokenizer.save_pretrained(ruta_local)
model.save_pretrained(ruta_local)

print(f"\nModelo guardado en: {ruta_local}")

In [ ]:
#Ver qué archivos se crearon
import os

print("Archivos guardados:")
for archivo in os.listdir(ruta_local):
    tamaño = os.path.getsize(os.path.join(ruta_local, archivo)) / 1024 / 1024  # MB
    print(f"  {archivo} ({tamaño:.2f} MB)")

### Ejemplo: Cargar desde local

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

#Cargar desde la ruta local (NO necesita internet)
ruta_local = "./modelos/mi_clasificador"

tokenizer_local = AutoTokenizer.from_pretrained(ruta_local)
model_local = AutoModelForSequenceClassification.from_pretrained(ruta_local)

print("Modelo cargado desde local")

#Usar en un pipeline
clasificador = pipeline("sentiment-analysis", model=model_local, tokenizer=tokenizer_local)

#Probar
resultado = clasificador("I love this movie!")
print(f"\nResultado: {resultado}")

---
## Ejercicio 1: Guarda y carga tu propio modelo

**Tareas:**
1. Elige un modelo diferente del Hub (sugerencias abajo)
2. Descárgalo y guárdalo localmente
3. Cárgalo desde local y úsalo

**Modelos sugeridos:**
- `nlptown/bert-base-multilingual-uncased-sentiment` (sentimiento en español)
- `dccuchile/bert-base-spanish-wwm-cased` (BERT español)
- `gpt2` (generación de texto)
- `google/flan-t5-small` (instrucciones)

In [ ]:
from transformers import AutoTokenizer, AutoModel, pipeline
import os

#TODO: Elige un modelo de HuggingFace
mi_modelo = ...  # Ej: "gpt2", "nlptown/bert-base-multilingual-uncased-sentiment"

#TODO: Define dónde guardarlo
mi_ruta = "./modelos/..."  # Ej: "./modelos/mi_gpt2"

#TODO: Descarga el tokenizer y modelo, usa la clase AutoModel correcta según el tipo de modelo


#TODO: Guarda localmente


#TODO: Lista los archivos guardados
print("\nArchivos guardados:")
for archivo in os.listdir(mi_ruta):
    print(f"  - {archivo}")

In [ ]:
#TODO: Carga el modelo desde local y úsalo

#Cargar
tokenizer_local = ...
model_local = ...

#Crear pipeline (ajusta la tarea según tu modelo)
#Ej: "sentiment-analysis", "text-generation", "fill-mask"
mi_pipeline = pipeline(..., model=model_local, tokenizer=tokenizer_local)

#Probar
resultado = mi_pipeline("Tu texto de prueba aquí")
print(resultado)

---
# Parte 2: Uso con GPU

Las GPUs aceleran enormemente la inferencia. Veamos cómo usarlas.

### Ejemplo: Verificar GPU disponible

In [ ]:
import torch

print("=== Información del sistema ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\n=== GPU detectada ===")
    print(f"Nombre: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("\nNo hay GPU disponible, se usará CPU")
    print("En Colab: Runtime → Change runtime type → GPU")

### Ejemplo: Cargar modelo en GPU

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

#Determinar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device}")

#Opción 1: Cargar modelo con device_map="auto" (automático)
model = AutoModelForCausalLM.from_pretrained("gpt2", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print(f"Modelo cargado en: {model.device}")

In [ ]:
#Opción 2: Especificar dispositivo en el pipeline
from transformers import pipeline
import torch

#device=0 significa GPU 0, device=-1 significa CPU
device_num = 0 if torch.cuda.is_available() else -1

generador = pipeline(
    "text-generation",
    model="gpt2",
    device=device_num
)

#Probar
resultado = generador("The future of AI is", max_new_tokens=20)
print(resultado[0]['generated_text'])

---
# Parte 3: Cuantización

Los modelos grandes (7B+ parámetros) no caben en GPUs normales. La **cuantización** reduce la precisión de los pesos para que ocupen menos memoria.

| Precisión | Bits | Memoria ~7B | Calidad |
|-----------|------|-------------|--------|
| float32 | 32 | ~28 GB | 100% |
| float16 | 16 | ~14 GB | ~99% |
| int8 | 8 | ~7 GB | ~97% |
| int4 | 4 | ~3.5 GB | ~93% |

### Ejemplo: Ver memoria de un modelo

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def mostrar_memoria_modelo(model, nombre="Modelo"):
    """Muestra la memoria que ocupa un modelo."""
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size = (param_size + buffer_size) / 1024**2  # MB
    
    print(f"{nombre}:")
    print(f"  Parámetros: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
    print(f"  Memoria: {total_size:.1f} MB")
    print(f"  Dtype: {next(model.parameters()).dtype}")
    return total_size

#Cargar modelo normal (float32)
modelo_normal = AutoModelForCausalLM.from_pretrained("gpt2")
mem_normal = mostrar_memoria_modelo(modelo_normal, "GPT-2 (float32)")

### Ejemplo: Cargar en float16 (half precision)

In [ ]:
#Cargar en float16 (mitad de memoria)
modelo_fp16 = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float16,
    device_map="auto"
)

mem_fp16 = mostrar_memoria_modelo(modelo_fp16, "\nGPT-2 (float16)")
print(f"\nReducción: {(1 - mem_fp16/mem_normal)*100:.0f}%")

### Ejemplo: Cuantización a 8 bits

In [ ]:
#Cuantización a 8 bits (requiere GPU)

if torch.cuda.is_available():
    modelo_8bit = AutoModelForCausalLM.from_pretrained(
        "gpt2",
        load_in_8bit=True,
        device_map="auto"
    )
    
    print("GPT-2 (8-bit):")
    print(f"  Cargado en: {modelo_8bit.device}")
    
    #Probar que funciona
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    inputs = tokenizer("Hello, I am", return_tensors="pt").to(modelo_8bit.device)
    outputs = modelo_8bit.generate(**inputs, max_new_tokens=20)
    print(f"  Output: {tokenizer.decode(outputs[0])}")
else:
    print("Cuantización 8-bit requiere GPU")

### Ejemplo: Cuantización a 4 bits

In [ ]:
from transformers import BitsAndBytesConfig

if torch.cuda.is_available():
    #Configuración para 4 bits
    config_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,  # Dtype para cálculos
        bnb_4bit_quant_type="nf4"  # Tipo de cuantización (nf4 es mejor)
    )
    
    modelo_4bit = AutoModelForCausalLM.from_pretrained(
        "gpt2",
        quantization_config=config_4bit,
        device_map="auto"
    )
    
    print("GPT-2 (4-bit):")
    print(f"  Cargado correctamente")
    
    #Probar
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    inputs = tokenizer("The meaning of life is", return_tensors="pt").to(modelo_4bit.device)
    outputs = modelo_4bit.generate(**inputs, max_new_tokens=20)
    print(f"  Output: {tokenizer.decode(outputs[0])}")
else:
    print("Cuantización 4-bit requiere GPU")

### Ejemplo: Comparar precisiones

Veamos cómo cambia la memoria y la calidad de las respuestas con diferentes precisiones.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

modelo_nombre = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
prompt = "Artificial intelligence will change the world by"

print(f"Prompt: {prompt}\n")
print("=" * 60)

#Float32
print("\n=== Float32 ===")
model_fp32 = AutoModelForCausalLM.from_pretrained(modelo_nombre)
mostrar_memoria_modelo(model_fp32, "Memoria")
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model_fp32.generate(**inputs, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id)
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
del model_fp32

#Float16
print("\n=== Float16 ===")
model_fp16 = AutoModelForCausalLM.from_pretrained(modelo_nombre, torch_dtype=torch.float16)
mostrar_memoria_modelo(model_fp16, "Memoria")
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model_fp16.generate(**inputs, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id)
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
del model_fp16

#8-bit (solo si hay GPU)
if torch.cuda.is_available():
    print("\n=== 8-bit ===")
    model_8bit = AutoModelForCausalLM.from_pretrained(modelo_nombre, load_in_8bit=True, device_map="auto")
    print(f"Memoria GPU usada: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
    inputs = tokenizer(prompt, return_tensors="pt").to(model_8bit.device)
    outputs = model_8bit.generate(**inputs, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id)
    print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
    del model_8bit
    torch.cuda.empty_cache()

---
## Ejercicio 2: Carga un modelo grande con cuantización

Vamos a cargar un modelo más grande usando cuantización para que quepa en memoria.

**Nota:** Este ejercicio requiere GPU.

**Modelos sugeridos:**
- `facebook/opt-1.3b` (~1.3B parámetros)
- `EleutherAI/pythia-1.4b` (~1.4B parámetros)
- `microsoft/phi-2` (~2.7B parámetros) - si tienes más memoria

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

if not torch.cuda.is_available():
    print("Este ejercicio requiere GPU")
else:
    #TODO: Elige un modelo grande
    modelo_grande = ...  # Ej: "facebook/opt-1.3b"
    
    #TODO: Configura cuantización 4-bit
    config_4bit = BitsAndBytesConfig(
        ...  #load_in_4bit, bnb_4bit_compute_dtype
    )
    
    print(f"Cargando {modelo_grande} en 4-bit...")
    
    #TODO: Cargar modelo y tokenizer
    model = ...
    tokenizer = ...
    
    print("Modelo cargado")
    print(f"Memoria GPU usada: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

In [ ]:
#TODO: Prueba el modelo con varios prompts

prompts = [
    "The meaning of life is",
    "Python is a programming language that",
    "In the year 2050, humans will",
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    
    #TODO: Generar respuesta
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Respuesta: {respuesta}")

---
# Resumen

| Técnica | Cuándo usarla | Comando clave |
|---------|---------------|---------------|
| **Guardar local** | Trabajar offline, deployment | `model.save_pretrained("./ruta")` |
| **GPU** | Acelerar inferencia | `device_map="auto"` o `device=0` |
| **float16** | Reducir memoria 50% | `torch_dtype=torch.float16` |
| **8-bit** | Modelos grandes | `load_in_8bit=True` |
| **4-bit** | Modelos muy grandes | `BitsAndBytesConfig(load_in_4bit=True)` |